<!-- # Create MOTEL TTL From motel-db

This notebook is a light workflow wrapper around the shared motel-db exporter.

Goal:
- generate the canonical TTL file used by this repository
- keep implementation logic in `app/ttl_creation/from_motel_db/generator_core.py`
- make the notebook readable, short, and publish-friendly

Current export includes:
- technology instances and mapped attributes
- process subclasses from `technology_variant`
- flow generation from motel-db `balancing`
- embedded carbon generation from motel-db linked entities
- unit and currency metadata sourced from motel-db `attribute.csv` -->


## Workflow

1. Set the motel-db input path and TTL output path.
2. Import `write_ttl_output(...)` from `generator_core.py`.
3. Run the export.
4. Review the summary counts and warnings.

The notebook intentionally does **not** duplicate the generator implementation.


In [1]:
from pathlib import Path

PATH_MOTEL_DB = Path(r"C:\Repositories\motel-platform\motel-db")
OUTPUT_TTL = Path(r"C:\Repositories\motel_ontology\app\data\01_classes_and_attributes\cls_atr_motel.ttl")

PATH_MOTEL_DB, OUTPUT_TTL

(WindowsPath('C:/Repositories/motel-platform/motel-db'),
 WindowsPath('C:/Repositories/motel_ontology/app/data/01_classes_and_attributes/cls_atr_motel.ttl'))

## Import Shared Generator

This cell makes sure the repository root is importable, then loads the canonical export function.


In [2]:
import sys

repo_root = Path.cwd()
if not (repo_root / "app").exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / "app").exists():
            repo_root = candidate
            break

repo_root_str = str(repo_root)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

from app.ttl_creation.from_motel_db.generator_core import write_ttl_output

## Run Export

This writes the canonical TTL file and returns a small summary dictionary.


In [3]:
result = write_ttl_output(
    path_motel_db=PATH_MOTEL_DB,
    output_ttl=OUTPUT_TTL,
    generated_by="ttl_creation_from_motel_db.ipynb",
)

ttl = result["ttl"]
generated_at = result["generated_at"]
warnings = result["warnings"]
stats = result["stats"]

summary = {
    "generated_at": generated_at,
    "output_ttl": str(OUTPUT_TTL),
    "n_lines": ttl.count("\n"),
    "n_triple_ends_approx": ttl.count(" ."),
    "n_instances_with_attributes_approx": ttl.count("dici_onto:hasAttribute"),
    "flows": stats.get("flows", 0),
    "embedded_carbon": stats.get("embedded_carbon", 0),
    "warnings": len(warnings),
}

summary

{'generated_at': '2026-06-27T00:16:13+02:00',
 'output_ttl': 'C:\\Repositories\\motel_ontology\\app\\data\\01_classes_and_attributes\\cls_atr_motel.ttl',
 'n_lines': 17840,
 'n_triple_ends_approx': 2877,
 'n_instances_with_attributes_approx': 2400,
 'flows': 322,
 'embedded_carbon': 119,
 'warnings': 483}

## Inspect Warnings

Warnings usually mean one of these:
- a motel-db attribute is not yet mapped to an ontology class
- a carrier lookup is missing
- an embedded-carbon record could not be linked to a technology-year instance

Showing only the first entries keeps the notebook light.


In [4]:
warnings[:20]

["LE_00001: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00002: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00003: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00004: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00005: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00006: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00007: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00008: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00009: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00010: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00011: unmapped attribute 'Annual Operational Expenditure Per Installed Capacity'",
 "LE_00012: unmapped 

## Notes

- The generated output is `app/data/01_classes_and_attributes/cls_atr_motel.ttl`.
- Backend/frontend behavior depends on reloading the updated TTL into the data store used by the app.
- If export logic changes are needed, update `generator_core.py` rather than adding notebook-only code.
